In [1]:
import os
import numpy as np, pandas as pd

ROOT = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
SRC  = os.path.join(ROOT, "zonal_features.parquet")
OUT  = os.path.join(ROOT, "zonal_features_fx.parquet")

HOT_DAY_C  = 24.0
CDH_BASE_C = 22.0

def add_fx(df):
    """Heat-wave memory features. Must be applied AFTER any weather swap."""
    df = df.sort_values(["zone", "utc"]).reset_index(drop=True)
    df["fx_app_roll72"] = (df.groupby("zone")["apparent_temperature"]
                             .transform(lambda s: s.rolling(72, min_periods=1).mean()))
    cdh = (df["apparent_temperature"] - CDH_BASE_C).clip(lower=0)
    df["fx_cdh24"] = cdh.groupby(df["zone"]).transform(lambda s: s.rolling(24, min_periods=1).sum())
    d = df["utc"].dt.date
    daily = df.assign(_d=d).groupby(["zone","_d"])["apparent_temperature"].max().rename("dmax").reset_index()
    daily["hot"] = daily["dmax"] >= HOT_DAY_C
    def streak(s):
        out, run = [], 0
        for h in s:
            run = run + 1 if h else 0
            out.append(run)
        return pd.Series(out, index=s.index)
    daily["fx_hot_streak_day"] = daily.groupby("zone")["hot"].transform(streak).astype(float)
    df = df.assign(_d=d).merge(daily[["zone","_d","fx_hot_streak_day"]], on=["zone","_d"], how="left").drop(columns="_d")
    return df

df = pd.read_parquet(SRC)
df["utc"] = pd.to_datetime(df["utc"])
before = list(df.columns)
df = add_fx(df)

FX = ["fx_app_roll72", "fx_cdh24", "fx_hot_streak_day"]
print("rows:", f"{len(df):,}", "| new cols:", [c for c in df.columns if c not in before])
print(df[FX].describe().round(2).to_string())
print("\nNaN counts:", {c: int(df[c].isna().sum()) for c in FX})
df.to_parquet(OUT)
print("\nsaved:", OUT)

rows: 1,291,433 | new cols: ['fx_app_roll72', 'fx_cdh24', 'fx_hot_streak_day']
       fx_app_roll72    fx_cdh24  fx_hot_streak_day
count     1291433.00  1291433.00         1291433.00
mean            8.02       19.34               3.45
std            12.37       43.23               9.83
min           -28.88        0.00               0.00
25%            -2.21        0.00               0.00
50%             7.60        0.00               0.00
75%            19.21       10.50               1.00
max            36.89      384.90              81.00

NaN counts: {'fx_app_roll72': 0, 'fx_cdh24': 0, 'fx_hot_streak_day': 0}

saved: /opt/app-root/src/Forecasting-Energy-Demand/Sangar/zonal_features_fx.parquet
